# Celeste AI - Multimodal Embeddings

Embed **text**, **images**, **video**, and **audio** into a unified vector space with `gemini-embedding-2-preview`.

Star on GitHub 👉 [withceleste/celeste-python](https://github.com/withceleste/celeste-python)

## Setup

In [ ]:
import os

import celeste
import numpy as np
from IPython.display import Image, display

---

## Text Embedding

Embed text using the `celeste.text` domain namespace.

In [ ]:
from dotenv import load_dotenv
load_dotenv("/Users/kamilbenkirane/Desktop/Projects/withceleste/celeste-python/.env")

text_result = await celeste.text.embed(
    "A happy golden retriever", model="gemini-embedding-2-preview", api_key=os.getenv("GOOGLE_API_KEY")
)

print(f"Dimensions: {len(text_result.content)}")
print(f"First 5 values: {text_result.content[:5]}")

---

## Image Embedding

Generate a dog image, then embed it using the `celeste.images` domain namespace.

In [ ]:
img_result = await celeste.images.generate(
    "A golden retriever dog", model="gemini-2.5-flash-image", api_key=os.getenv("GOOGLE_API_KEY")
)
display(Image(data=img_result.content.data))

In [ ]:
img_emb = await celeste.images.embed(
    img_result.content, model="gemini-embedding-2-preview", api_key=os.getenv("GOOGLE_API_KEY")
)
print(f"Dimensions: {len(img_emb.content)}")
print(f"First 5 values: {img_emb.content[:5]}")

---

## Video Embedding

Download a short sample video and embed it using the `celeste.videos` domain namespace.

In [ ]:
import httpx
from celeste.artifacts import VideoArtifact
from celeste.mime_types import VideoMimeType
from IPython.display import Video, Audio

video_bytes = httpx.get("https://download.samplelib.com/mp4/sample-5s.mp4").content
video = VideoArtifact(data=video_bytes, mime_type=VideoMimeType.MP4)
display(Video(data=video_bytes, embed=True, mimetype="video/mp4"))

vid_emb = await celeste.videos.embed(video, model="gemini-embedding-2-preview", api_key=os.getenv("GOOGLE_API_KEY"))
print(f"Dimensions: {len(vid_emb.content)}")
print(f"First 5 values: {vid_emb.content[:5]}")

---

## Audio Embedding

Download a short sample audio and embed it using the `celeste.audio` domain namespace.

In [ ]:
from celeste.artifacts import AudioArtifact
from celeste.mime_types import AudioMimeType

audio_bytes = httpx.get("https://download.samplelib.com/mp3/sample-3s.mp3").content
audio = AudioArtifact(data=audio_bytes, mime_type=AudioMimeType.MP3)
display(Audio(data=audio_bytes, autoplay=False))

aud_emb = await celeste.audio.embed(audio, model="gemini-embedding-2-preview", api_key=os.getenv("GOOGLE_API_KEY"))
print(f"Dimensions: {len(aud_emb.content)}")
print(f"First 5 values: {aud_emb.content[:5]}")

---

## Multimodal Search Test

Query: dog image. Compare similarity to text "dog" vs text "chair".

In [ ]:
!uv pip install matplotlib seaborn

In [ ]:
import seaborn as sns
import pandas as pd


def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


texts = ["dog", "labrador", "chair"]
embeddings = {
    t: await celeste.text.embed(t, model="gemini-embedding-2-preview") for t in texts
}
scores = {t: cosine_sim(img_emb.content, emb.content) for t, emb in embeddings.items()}

df = pd.DataFrame({"text": scores.keys(), "similarity": scores.values()}).sort_values(
    "similarity", ascending=False
)
sns.barplot(data=df, x="similarity", y="text").set(
    xlim=(0, 1), title="Image(dog) similarity to:"
)

assert scores["dog"] > scores["chair"]

---
Star on GitHub 👉 [withceleste/celeste-python](https://github.com/withceleste/celeste-python)